# Metaflow Tutorial: Combining Parameters, Files, and Foreach

This notebook demonstrates a combined workflow using elements from Metaflow Tutorials 2 and 3.
- **From Tutorial 2**: Ingesting external data via `IncludeFile` and using `Parameter` for configuration.
- **From Tutorial 3**: Using the `foreach` keyword to dynamically parallelize analytics over multiple categories (genres).

### 1. Setup
Load the Metaflow Jupyter extension.

In [1]:
%load_ext metaflow_jupyter

### 2. Global Helpers & Imports
We'll define a helper to find our CSV path. Note that we **don't** need to import `Parameter` or `IncludeFile` manually — our registry will detect them and add them to the imports automatically!

In [2]:
%%mf_import MovieStats
import csv
from random import shuffle

In [3]:
%%mf_global MovieStats
def script_path(filename):
    import os
    filepath = os.path.join(os.getcwd())
    return os.path.join(filepath, filename)

### 3. Local State (Parameters & Files)
Here we define our inputs.

In [4]:
%%mf_local MovieStats files 
movie_data = IncludeFile(
    "movie_data",
    help="The path to a movie metadata file.",
    default=script_path("movies.csv"),
)

### 4. The Flow Steps
We start by parsing and then fan-out using `foreach`.

In [5]:
%%mf_step MovieStats.start
columns = ["movie_title", "title_year", "genres", "gross"]
self.dataframe = {col: [] for col in columns}

In [6]:
%%mf_step MovieStats.start dataframe_make
for row in csv.DictReader(self.movie_data.splitlines()):
    for col in columns:
        val = int(row[col]) if col in ("title_year", "gross") else row[col]
        self.dataframe[col].append(val)

In [7]:
%%mf_step MovieStats.start genre_make
self.genres = list({g for gs in self.dataframe["genres"] for g in gs.split("|")})[:5]
print(f"Fanning out over genres: {self.genres}")

In [8]:
%%mf_step MovieStats.compute_statistics
self.genre = self.input
selector = [self.genre in row for row in self.dataframe["genres"]]

self.genre_movies = [
    title for title, is_match in zip(self.dataframe["movie_title"], selector) 
    if is_match
]
shuffle(self.genre_movies)
self.next(self.join)

In [9]:
%%mf_local MovieStats params
top_n = Parameter(
    "top_n",
    help="Number of movies to show in summary",
    default=3
)

In [10]:
%%mf_step MovieStats.join join
self.results = {
    inp.genre: inp.genre_movies[:self.top_n]
    for inp in inputs
}
self.next(self.end)

In [11]:
%%mf_step MovieStats.end
for genre, movies in self.results.items():
    print(f"\nGenre: {genre}")
    for i, m in enumerate(movies, 1):
        print(f"  {i}. {m}")

In [12]:
%%mf_step MovieStats.start end_start
self.next(self.compute_statistics, foreach="genres")

### 5. Run the Analytics

In [13]:
run = %mf_run MovieStats

Metaflow 2.19.20 executing MovieStats for user:sam900
Validating your flow...
    The graph looks good!
Running pylint...
    Pylint not found, so extra checks are disabled.
Including file /home/sam900/metaflow_project/metaflow-jupyter/movies.csv of size 611B 
2026-03-12 04:07:49.897 Workflow starting (run-id 1773288469887751):
2026-03-12 04:07:49.964 [1773288469887751/start/1 (pid 13394)] Task is starting.
2026-03-12 04:07:50.804 [1773288469887751/start/1 (pid 13394)] Fanning out over genres: ['Crime', 'Musical', 'Comedy', 'Adventure', 'Fantasy']
2026-03-12 04:07:51.036 [1773288469887751/start/1 (pid 13394)] Foreach yields 5 child steps.
2026-03-12 04:07:51.036 [1773288469887751/start/1 (pid 13394)] Task finished successfully.
2026-03-12 04:07:51.055 [1773288469887751/compute_statistics/2 (pid 13409)] Task is starting.
2026-03-12 04:07:51.063 [1773288469887751/compute_statistics/3 (pid 13410)] Task is starting.
2026-03-12 04:07:51.075 [1773288469887751/compute_statistics/4 (pid 13411)

In [14]:
%mf_clear -all

Metaflow Registry: Cleared all flows.
